In [0]:
# # ============================================================== 
# # Silver Layer Orchestrator with Multiple Table Processing
# # Purpose:
# #     This script runs the Silver layer ETL pipeline for multiple tables,
# #     orchestrating the entire pipeline with detailed logging, error handling,
# #     and metrics tracking.
# #
# # Key Features:
# # - Processes multiple Bronze tables with configurable rules.
# # - Applies data quality checks, deduplication, and updates Silver & audit tables.
# # - Metrics and logging for every table processed.
# # ============================================================== 

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

import os
env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_silver")

from superstore_logger import get_superstore_logger, log_event
from superstore_orchestrator_execution_context import get_execution_context
from superstore_platform_constants import SILVER_LAYER
from superstore_platform_config import table, get_bronze_schema, get_silver_schema, get_quarantine_schema, get_audit_schema, get_metrics_schema

execution_context = get_execution_context(dbutils, allow_standalone=True)
master_run_id = execution_context["master_run_id"]
layer_run_id = execution_context["layer_run_id"]

logger_silver = get_superstore_logger("superstore_silver_layer_orchestrator")

log_event(
    logger_silver,
    "INFO",
    "Superstore Silver layer ETL orchestrator started",
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=SILVER_LAYER
)

import importlib
import superstore_silver_module
importlib.reload(superstore_silver_module)
from superstore_silver_module import ( bronze_to_silver_prod, write_etl_metrics)

# -------------------------------
# Load Silver table configuration 
# -------------------------------
import yaml
config_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/configs/superstore_silver_config/superstore_silver_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

silver_table_configs_raw = config["silver_table_configs"]

# -------------------------------
# Dynamically resolve table paths per environment
# -------------------------------
silver_table_configs = []
for cfg in silver_table_configs_raw:
    silver_table_configs.append({
        "bronze_table": table(get_bronze_schema(), cfg["bronze_table"].split('.')[-1]),
        "silver_table": table(get_silver_schema(), cfg["silver_table"].split('.')[-1]),
        "quarantine_table": table(get_quarantine_schema(), cfg["quarantine_table"].split('.')[-1]),
        "audit_table": table(get_audit_schema(), cfg["audit_table"].split('.')[-1]),
        "read_from_table": table(get_bronze_schema(), cfg["read_from_table"].split('.')[-1]),
        "business_keys": cfg["business_keys"],
        "numeric_cast_cols": cfg.get("numeric_cast_cols", {}),
        "regex_cols": cfg.get("regex_cols", {}),
        "categorical_allowed_vals": cfg.get("categorical_allowed_vals", {}),
        "metrics_table": table(get_metrics_schema(), cfg["metrics_table"].split('.')[-1]),
        "layer_name": cfg.get("layer_name", "silver"),
        "shuffle_partitions": cfg.get("shuffle_partitions", 200)
    })

# -------------------------------
# Bronze-Silver ETL wrapper
# -------------------------------
def process_multiple_tables(
    spark, table_configs: list, master_run_id: str
):
    log_event(
        logger_silver,
        "INFO",
        "Starting to process multiple Bronze layer table rows to Silver tables",
        total_tables=len(table_configs),
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=SILVER_LAYER
    )

    for cfg in table_configs:
        bronze_table = cfg["bronze_table"]

        log_event(
            logger_silver,
            "INFO",
            "Processing Bronze table",
            table=bronze_table,
            master_run_id=master_run_id,
            layer_run_id=layer_run_id,
            layer=SILVER_LAYER
        )

        try:
            metrics = bronze_to_silver_prod(
                spark,
                bronze_table=cfg["bronze_table"],
                silver_table=cfg["silver_table"],
                audit_table=cfg["audit_table"],
                business_keys=cfg["business_keys"],
                numeric_cast_cols=cfg["numeric_cast_cols"],
                regex_cols=cfg["regex_cols"],
                categorical_allowed_vals=cfg["categorical_allowed_vals"],
                quarantine_table=cfg["quarantine_table"],
                shuffle_partitions=cfg["shuffle_partitions"],
                layer_name=cfg["layer_name"],
                metrics_table=cfg["metrics_table"],
                master_run_id=master_run_id,
                layer_run_id=layer_run_id
            )

            if metrics.get("metrics_table"):
                write_etl_metrics(
                    spark,
                    metrics_table=metrics["metrics_table"],
                    layer_run_id=metrics["layer_run_id"],
                    master_run_id=metrics["master_run_id"],
                    layer_name=metrics["layer"],
                    read_from_table=cfg["read_from_table"],
                    table_name=metrics["table_name"],
                    start_ts=metrics["start_ts"],
                    end_ts=metrics["end_ts"],
                    total_rows=metrics["total_rows"],
                    good_rows=metrics["good_rows"],
                    dirty_rows=metrics["dirty_rows"],
                    deduplicated_rows=metrics["deduplicated_rows"],
                    duplicate_rows=metrics["duplicate_rows"],
                    throughput_rows_per_sec=metrics["throughput_rows_per_sec"],
                    skew_ratio=metrics["skew_ratio"],
                    run_status=metrics["run_status"],
                    duration_secs=metrics["duration_secs"],
                    notes=metrics["notes"]
                )

            log_event(
                logger_silver,
                "INFO",
                "Completed ETL for table",
                table=bronze_table,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=SILVER_LAYER
            )

        except Exception as e:
            log_event(
                logger_silver,
                "ERROR",
                f"ETL failed for table: {bronze_table}",
                table=bronze_table,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=SILVER_LAYER,
                error=str(e)
            )

    log_event(
        logger_silver,
        "INFO",
        "Finished processing Bronze to Silver ETL for all tables",
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=SILVER_LAYER
    )

# -------------------------------
# Execute Bronze → Silver ETL for multiple tables
# -------------------------------
if __name__ == "__main__":
    process_multiple_tables(spark, silver_table_configs, master_run_id)

